# Notebook 09 — Final Evaluation Tables

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Stage:** 9 of 10

## Purpose

Pulls every result from Notebooks 01-08 into paper-ready tables and one summary figure. No new computation — just curation. **Runs on CPU (no GPU needed).**

## Tables produced

| Table | Content |
|---|---|
| paper_table_1_performance | Accuracy / Macro-F1 / MCC for all 6 models on both datasets |
| paper_table_2_calibration | ECE before/after for all 6 models on both datasets |
| paper_table_3_stability | Jaccard top-10 per (model, perturbation) — NSL-KDD |
| paper_table_4_krishna | Krishna 6 metrics at k=10 (NSL-KDD) |
| paper_table_5_scts | SCTS-v2 mean + bottom/top quartile accuracy gap |
| paper_table_6_llm | LLM alert quality scores (Faithfulness/Coherence/Actionability) |
| paper_table_7_cross_dataset | Side-by-side NSL-KDD vs CIC-IDS2017 summary |

## Figures produced

| Figure | Content |
|---|---|
| paper_figure_summary | 4-panel: calibration, stability, SCTS-v2, LLM scores |

## Output

```
results/paper_tables/    ← all the above CSVs + figure
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

PAPER_DIR = Path(REPO) / 'results' / 'paper_tables'
PAPER_DIR.mkdir(parents=True, exist_ok=True)

TABLES = Path(REPO) / 'results' / 'tables'
MODELS_NSL = Path(REPO) / 'models' / 'nsl_kdd'
MODELS_CIC = Path(REPO) / 'models' / 'cic_ids2017'

CANONICAL = ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
             'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']

print('Output paper-tables directory:', PAPER_DIR)

---
## Table 1 — Performance summary (Acc / Macro-F1 / MCC)

In [ ]:
rows = []

# NSL-KDD: load from models/nsl_kdd/metrics.json (covers all 12 models from Notebook 02)
with open(MODELS_NSL / 'metrics.json') as f:
    nsl_metrics = json.load(f)
for name in CANONICAL:
    if name in nsl_metrics:
        m = nsl_metrics[name]
        rows.append({
            'Dataset': 'NSL-KDD', 'Model': name,
            'Accuracy': m['accuracy'], 'MacroF1': m['f1_macro'], 'MCC': m['mcc'],
        })

# CIC-IDS2017
if (MODELS_CIC / 'metrics.json').exists():
    with open(MODELS_CIC / 'metrics.json') as f:
        cic_metrics = json.load(f)
    for name in CANONICAL:
        if name in cic_metrics:
            m = cic_metrics[name]
            rows.append({
                'Dataset': 'CIC-IDS2017', 'Model': name,
                'Accuracy': m['accuracy'], 'MacroF1': m['f1_macro'], 'MCC': m['mcc'],
            })
else:
    print('⚠ CIC-IDS2017 metrics.json not found — only NSL-KDD will be in Table 1')

df1 = pd.DataFrame(rows)
df1.to_csv(PAPER_DIR / 'paper_table_1_performance.csv', index=False)
print('TABLE 1 — Performance')
print(df1.to_string(index=False, float_format='%.4f'))

---
## Table 2 — Calibration impact (ECE before/after)

In [ ]:
rows = []

# NSL-KDD — load from calibration v2 table
nsl_calib_path = TABLES / 'nslkdd_calibration_v2.csv'
if not nsl_calib_path.exists():
    # Fallback to original
    nsl_calib_path = TABLES / 'nslkdd_calibration.csv'

if nsl_calib_path.exists():
    df_nsl = pd.read_csv(nsl_calib_path)
    for _, r in df_nsl.iterrows():
        rows.append({
            'Dataset': 'NSL-KDD',
            'Model': r.get('Model', r.get('model', '?')),
            'ECE_before': r.get('ECE_before', r.get('ECE_uncalibrated', None)),
            'ECE_after': r.get('ECE_after', r.get('ECE_calibrated', None)),
        })

# CIC-IDS2017
cic_calib_path = TABLES / 'cic_calibration.csv'
if cic_calib_path.exists():
    df_cic = pd.read_csv(cic_calib_path)
    for _, r in df_cic.iterrows():
        rows.append({
            'Dataset': 'CIC-IDS2017',
            'Model': r['Model'],
            'ECE_before': r['ECE_before'],
            'ECE_after': r['ECE_after'],
        })

df2 = pd.DataFrame(rows)
df2['ECE_reduction_%'] = 100 * (df2['ECE_before'] - df2['ECE_after']) / df2['ECE_before'].clip(lower=1e-9)
df2.to_csv(PAPER_DIR / 'paper_table_2_calibration.csv', index=False)
print('TABLE 2 — Calibration')
print(df2.to_string(index=False, float_format='%.4f'))

---
## Table 3 — Stability (Jaccard top-10 per perturbation, NSL-KDD)

In [ ]:
stab_path = TABLES / 'nslkdd_stability.csv'
if stab_path.exists():
    df3 = pd.read_csv(stab_path)
    df3.to_csv(PAPER_DIR / 'paper_table_3_stability.csv', index=False)
    print('TABLE 3 — Stability (NSL-KDD)')
    print(df3.to_string(index=False, float_format='%.4f'))
else:
    print('⚠ nslkdd_stability.csv not found')

---
## Table 4 — Krishna metrics (k=10, NSL-KDD)

In [ ]:
kr_path = TABLES / 'nslkdd_krishna_summary.csv'
if kr_path.exists():
    df4 = pd.read_csv(kr_path)
    df4.to_csv(PAPER_DIR / 'paper_table_4_krishna.csv', index=False)
    print('TABLE 4 — Krishna cross-model agreement (NSL-KDD, k=10)')
    print(df4.to_string(index=False, float_format='%.3f'))
else:
    print('⚠ Krishna summary not found')

---
## Table 5 — SCTS-v2 trust score (mean + accuracy gap)

In [ ]:
scts_summary_path = TABLES / 'nslkdd_scts_summary.csv'
scts_val_path = TABLES / 'nslkdd_scts_validation.csv'

rows = []
if scts_summary_path.exists():
    df_scts = pd.read_csv(scts_summary_path)
    # Mean SCTS per model (across all classes)
    mean_scts = df_scts.groupby('Model').apply(
        lambda d: pd.Series({
            'mean_SCTS': (d['mean_SCTS'] * d['n']).sum() / d['n'].sum(),
            'mean_c1': (d['mean_c1'] * d['n']).sum() / d['n'].sum(),
            'mean_c2': (d['mean_c2'] * d['n']).sum() / d['n'].sum(),
            'mean_c3': (d['mean_c3'] * d['n']).sum() / d['n'].sum(),
        })
    ).reset_index()

    # Get accuracy gap from validation table
    if scts_val_path.exists():
        df_val = pd.read_csv(scts_val_path)
        # Get bottom and top quartile accuracies per model
        for _, row in mean_scts.iterrows():
            name = row['Model']
            sub = df_val[df_val['Model'] == name]
            bot = sub[sub['SCTS_low'] == 25]
            top = sub[sub['SCTS_low'] == 75]
            bot_acc = float(bot['accuracy'].iloc[0]) if len(bot) > 0 else None
            top_acc = float(top['accuracy'].iloc[0]) if len(top) > 0 else None
            gap_pp = (top_acc - bot_acc) * 100 if (bot_acc and top_acc) else None
            rows.append({
                'Model': name,
                'mean_SCTS': row['mean_SCTS'],
                'mean_c1_calibration': row['mean_c1'],
                'mean_c2_stability': row['mean_c2'],
                'mean_c3_conformal': row['mean_c3'],
                'bottom_quartile_acc': bot_acc,
                'top_quartile_acc': top_acc,
                'accuracy_gap_pp': gap_pp,
            })

df5 = pd.DataFrame(rows)
df5.to_csv(PAPER_DIR / 'paper_table_5_scts.csv', index=False)
print('TABLE 5 — SCTS-v2 trust score')
print(df5.to_string(index=False, float_format='%.3f'))

---
## Table 6 — LLM alert quality

In [ ]:
llm_path = TABLES / 'nslkdd_llm_alerts_summary.csv'
if llm_path.exists():
    df6 = pd.read_csv(llm_path)
    df6.to_csv(PAPER_DIR / 'paper_table_6_llm.csv', index=False)
    print('TABLE 6 — LLM alert quality (Llama-3 generator + judge)')
    print(df6.to_string(index=False, float_format='%.2f'))
else:
    print('⚠ LLM summary not found')

---
## Table 7 — Cross-dataset summary

In [ ]:
# Compact one-row-per-(dataset, model) summary with the headline numbers
rows = []

# NSL-KDD
if 'df_nsl' in dir() or nsl_calib_path.exists():
    df_nsl_calib = pd.read_csv(nsl_calib_path)
    for name in CANONICAL:
        m = nsl_metrics.get(name, {})
        cal = df_nsl_calib[df_nsl_calib.iloc[:, 0] == name]  # match Model column position
        if len(cal) == 0:
            cal_row = {}
        else:
            cal_row = cal.iloc[0].to_dict()
        ece_before = cal_row.get('ECE_before') or cal_row.get('ECE_uncalibrated')
        ece_after = cal_row.get('ECE_after') or cal_row.get('ECE_calibrated')
        rows.append({
            'Dataset': 'NSL-KDD',
            'Model': name,
            'Accuracy': m.get('accuracy'),
            'MCC': m.get('mcc'),
            'ECE_before': ece_before,
            'ECE_after': ece_after,
        })

# CIC-IDS2017
if (MODELS_CIC / 'metrics.json').exists():
    df_cic_calib = pd.read_csv(TABLES / 'cic_calibration.csv') if (TABLES / 'cic_calibration.csv').exists() else None
    for name in CANONICAL:
        m = cic_metrics.get(name, {})
        if df_cic_calib is not None:
            sub = df_cic_calib[df_cic_calib['Model'] == name]
            ece_before = sub['ECE_before'].iloc[0] if len(sub) else None
            ece_after  = sub['ECE_after'].iloc[0] if len(sub) else None
        else:
            ece_before = ece_after = None
        rows.append({
            'Dataset': 'CIC-IDS2017',
            'Model': name,
            'Accuracy': m.get('accuracy'),
            'MCC': m.get('mcc'),
            'ECE_before': ece_before,
            'ECE_after': ece_after,
        })

df7 = pd.DataFrame(rows)
df7.to_csv(PAPER_DIR / 'paper_table_7_cross_dataset.csv', index=False)
print('TABLE 7 — Cross-dataset summary')
print(df7.to_string(index=False, float_format='%.4f'))

---
## Headline figure — 4-panel summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Calibration impact across both datasets (ECE before/after)
ax = axes[0, 0]
df_p2 = df2.copy()
df_p2['label'] = df_p2['Dataset'] + '/' + df_p2['Model'].str.replace('_', '')
x = np.arange(len(df_p2))
w = 0.4
ax.bar(x - w/2, df_p2['ECE_before'], width=w, label='Before', color='#DD8452')
ax.bar(x + w/2, df_p2['ECE_after'], width=w, label='After', color='#4C72B0')
ax.set_xticks(x); ax.set_xticklabels(df_p2['label'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('ECE'); ax.set_title('Calibration impact'); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_yscale('log')

# Panel 2: Stability — mean Jaccard@10 per (model, perturbation) on NSL-KDD
ax = axes[0, 1]
if stab_path.exists():
    df_st = pd.read_csv(stab_path)
    pivot = df_st.pivot(index='Model', columns='Perturbation', values='Jaccard@10')
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar_kws={'label': 'Jaccard@10'}, ax=ax)
    ax.set_title('Explanation stability (NSL-KDD)')
else:
    ax.text(0.5, 0.5, 'Stability data not found', ha='center', va='center', transform=ax.transAxes)

# Panel 3: SCTS-v2 accuracy gap (bottom vs top quartile)
ax = axes[1, 0]
if 'df5' in dir() and not df5.empty and 'accuracy_gap_pp' in df5.columns:
    df_p5 = df5.dropna(subset=['accuracy_gap_pp']).sort_values('accuracy_gap_pp')
    ax.barh(df_p5['Model'], df_p5['accuracy_gap_pp'], color='#4C72B0')
    ax.set_xlabel('Top-vs-bottom quartile accuracy gap (pp)')
    ax.set_title('SCTS-v2: accuracy stratification by trust quartile')
    ax.grid(axis='x', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'SCTS data not found', ha='center', va='center', transform=ax.transAxes)

# Panel 4: LLM alert scores
ax = axes[1, 1]
if llm_path.exists():
    df_p6 = pd.read_csv(llm_path)
    metrics = ['Faithfulness', 'Coherence', 'Actionability']
    x = np.arange(len(df_p6))
    width = 0.25
    for i, metric in enumerate(metrics):
        if metric in df_p6.columns:
            ax.bar(x + (i-1)*width, df_p6[metric], width=width, label=metric)
    ax.set_xticks(x); ax.set_xticklabels(df_p6['Model'], rotation=45, ha='right', fontsize=7)
    ax.set_ylim(0, 5.2); ax.set_ylabel('Score (1-5)')
    ax.set_title('LLM alert quality (Llama-3 judge)')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'LLM data not found', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(PAPER_DIR / 'paper_figure_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save all + commit

In [ ]:
print('Paper deliverable files:')
for f in sorted(PAPER_DIR.iterdir()):
    sz = f.stat().st_size / 1024
    print(f'  {f.name:<40}  {sz:>8.1f} KB')

os.chdir(REPO)
!git add notebooks/09_final_evaluation.ipynb
!git add results/paper_tables/
!git status --short
!git commit -m 'Notebook 09: final paper-ready tables and summary figure'
!git push